In [3]:
import os
import re
import pandas as pd
from tqdm import tqdm
import json
import requests
from datetime import datetime
import time

In [4]:
# Openrouter configuration
OPENROUTER_API_KEY = "Insert your own api key here"

# define prompt and functions

#### 1. test gemma 3 27b

In [9]:
# Error log for sentiment comment generation
error_log_file_sentiment = "sentiment_error_log.txt"


def openrouter_generate_comments(paragraph, n_comments=10):
    """Call moonshotai/kimi-k2:free to generate synthetic comments with sentiment labels.

    Returns raw text from the model (one comment per line if it follows the prompt).
    Logs any API errors to error_log_file_sentiment.
    """
    system_prompt = (
        "You are an expert in sentiment analysis, deeply familiar with Chinese web novel "
        "communities, internet culture, buzzwords, catchphrases, memes, and the use of "
        "emoticons/slang in online discussions. Your task is to generate comments that "
        "authentically reflect the style, tone, and emotional nuance of Chinese web novel readers."
    )

    user_prompt = f"""
Task: Read the provided paragraph and generate {n_comments} Chinese (written) comments that fit the style of Chinese web novel readers. 
The comments should be a balanced mix of positive, negative, and neutral sentiments, as defined below.

Definitions:
Positive: The text contains explicit or implicit clues suggesting the speaker is (predominantly) in a positive emotional state (e.g., happy, amused, admiring, relaxed, forgiving, entertained, appreciative, thankful, excited, playful, humorous, joking, optimistic, inspired, or using supportive/encouraging language).
Negative: The text contains explicit or implicit clues suggesting the speaker is (predominantly) in a negative emotional state (e.g., sad, angry, frustrated, anxious, violent, disappointed, pessimistic, critical, ironic, sarcastic, or using mocking/derogatory language).
Neutral: The text primarily states facts, observations, or balanced/mixed emotions without a clear positive or negative lean. May include speculative, analytical, or matter-of-fact statements.

Special Considerations:
Sarcasm/irony/mockery vs. Playful/humorous/joking:
- Negative sarcasm/mockery: Cutting, dismissive, or hostile remarks that belittle (e.g., "Oh great, another brilliant idea").
- Positive joking/playfulness: Humor intended to bond, lighten mood, or show affection (e.g., "Here comes the shining scape goat lol").

Emoticons/Slang: Use emoticons and internet slang to enhance emotional expression.
Pop Culture/Memes: Reference well-known memes, tropes, or catchphrases from Chinese web novels, games, or online culture.
Character/Plot Analysis: Include comments that analyze character motivations, plot twists, or authorial intent, as these are common in web novel discussions.

Output Format:
For each generated comment, provide exactly ONE line in this format (no extra text, no numbering, no bullets):
SyntheticText: [comment text]    Label: [positive/negative/neutral]

comment samples: 
起誓人也是这么想的，快被黑星逼疯了 Neutral 
然后发现除了自己其他都是二五仔 Neutral 
完了，又一个姬家。 Negative 
红色苏联那位？ Neutral 
暗暗做出决定×2 Neutral 
"呯——"好了，现在大家都是自己人了 Neutral 
星龙的好帮手 Positive 最后发现你是琴酒！ Neutral 
同志，萌芽内务部！你被捕了！ Negative 
每次这种竞争者桥段真的。。。。，又是必输的，还经常有，这是强行制造矛盾冲突吗？只能说水字数又弱智。一号，马杰，还有这什么螺旋，每十章出一个是吧？好伤阅读体验啊( ˘•ω•˘ ) Negative 
要么协作，要么内耗。 Neutral 
现实这种人太多，就那袁老来说，早期研究杂交水稻的时候被同行各种打压，造谣抹黑，他老人家走后还有一群人欢呼，在外网上的评论看多了，真的会脑溢血。。 Negative 

Paragraph for Comment Generation:
{paragraph}
"""

    try:
        time.sleep(60)
        url = "https://openrouter.ai/api/v1/chat/completions"
        
        response = requests.post(
            url=url,
            headers={"Authorization": "Bearer Insert your own api key here"},
            json={
                "model": "google/gemma-3-27b-it:free",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": 0.9,
                "max_tokens": 2000,
            },
            timeout=60,
        )
        print("Rate Limit Headers:")
        print(f"  Limit: {response.headers.get('x-ratelimit-limit')}")
        print(f"  Remaining: {response.headers.get('x-ratelimit-remaining')}")
        print(f"  Reset: {response.headers.get('x-ratelimit-reset')}")

        response.raise_for_status()
        response_data = response.json()
        response_content = response_data["choices"][0]["message"]["content"]
        # Do NOT clean line breaks; we want one comment per line
        return response_content

    except requests.exceptions.HTTPError as e:
        # Enhanced error logging
        print(f"\n=== 429 ERROR DETAILS ===")
        print(f"Status: {e.response.status_code}")
        print(f"Headers: {dict(e.response.headers)}")
        print(f"Body: {e.response.text}")
        print(f"=========================\n")
        
        with open(error_log_file_sentiment, "a", encoding="utf-8") as log_f:
            log_f.write(
                f"[{datetime.now().isoformat()}] ERROR: {repr(e)}\n"
                f"Response: {e.response.text}\n"
                f"Headers: {dict(e.response.headers)}\n\n"
            )
        return None
    
    except Exception as e:
        # Log any error to file with timestamp and short paragraph snippet
        with open(error_log_file_sentiment, "a", encoding="utf-8") as log_f:
            log_f.write(
                f"[{datetime.now().isoformat()}] ERROR during openrouter_generate_comments: {repr(e)}\n"
            )
        print(f"Error during API call: {e}")
        
        
        return None


In [10]:
# Example: generate synthetic comments for one paragraph
paragraph = """
[君邪突然醒了过来。 他甚至还不等睁开眼睛，下意识的右手一拍地面，就要跃起身来。此乃是非之地，生死一发，不可久留]
"""

raw_output = openrouter_generate_comments(paragraph, n_comments=20)

if raw_output is None:
    print("LLM call failed; see sentiment_error_log.txt for details.")
else:
    print(raw_output[:2000])  # preview first 2000 characters

Rate Limit Headers:
  Limit: None
  Remaining: None
  Reset: None

SyntheticText: 卧槽！君邪这反应速度，是练过的吧？起码是宗师级别！ Positive
SyntheticText: 果然不出所料，又到了跑路的时候。君邪这人设就是坑货，永远在刀尖上跳舞。 Negative
SyntheticText: 这个作者就喜欢这种一上来就绝境的开局，刺激归刺激，能不能换个花样？ Neutral
SyntheticText: 这“生死一发”的描述，好熟悉，感觉跟上一部升级流小说一个模子刻出来的。 Negative
SyntheticText: 君邪醒了！剧情终于要开始了！激动的心，颤抖的手！ Positive
SyntheticText: 下意识反应，说明他骨子里是个惜命的人啊，哈哈哈哈。 Playful
SyntheticText: 拍地面的动作好帅啊！有种霸道总裁的感觉！ Positive
SyntheticText: 感觉下一秒就要开始打脸了，就问你一句：敢不敢站出来？ Neutral
SyntheticText: 这种地方，不跑留着过年吗？作者你倒是写啊！ Negative
SyntheticText: 君邪，你可长点心吧！总是这样，读者都替你着急！ Positive
SyntheticText: “不可久留”四个字写得真好，简洁明了，充满了紧迫感。 Positive
SyntheticText: 看到“君邪”，直接想起了那句“君要臣死，臣不得不死”。 Neutral
SyntheticText: 这种剧情，我猜他肯定是被追杀的，然后要开始隐藏身份，装猪吃老虎。 Neutral
SyntheticText: 难道是老熟敌又来搞事了？希望这次别又是主角光环开得太大。 Negative
SyntheticText: 作者，别再让君邪背锅了！他已经够惨了！ Negative
SyntheticText: 救命啊！又一个苦命的男主！感觉他这辈子都要经历各种磨难了。 Negative
SyntheticText: 感觉这个“是非之地”绝对不简单，后面肯定有大boss等着他。 Neutral
SyntheticText: 君邪这手速，我酸了！求君邪的训练方法！ Playful
SyntheticText

Model: google/gemma-3-27b-it:free
generate synthetic data evaluation:

positive 6; 1 out of 6 is wrongly labeled

negative 6; all correct

neutral 6; 3 out of 6 is wrongly labeled

"playful" shouldn't be a tag 

### refine the prompt to v2.0 with chinese annotated datasheet prompt A / paragraph from bookId 1321320

In [14]:
# Error log for sentiment comment generation
error_log_file_sentiment = "sentiment_error_log.txt"


def openrouter_generate_comments(paragraph, n_comments=10):
    """Call moonshotai/kimi-k2:free to generate synthetic comments with sentiment labels.

    Returns raw text from the model (one comment per line if it follows the prompt).
    Logs any API errors to error_log_file_sentiment.
    """
    system_prompt = (
        "You are an expert in sentiment analysis, deeply familiar with Chinese web novel "
        "communities, internet culture, buzzwords, catchphrases, memes, and the use of "
        "emoticons/slang in online discussions. Your task is to generate comments that "
        "authentically reflect the style, tone, and emotional nuance of Chinese web novel readers."
    )

    user_prompt = f"""
Task: Read the provided paragraph and generate {n_comments} Chinese (written) comments that fit the style of Chinese web novel readers. 
The comments should be a balanced number of positive or negative comments, as defined below.

Definitions:
Positive: The text contains explicit or implicit clues suggesting the speaker is (predominantly) in a positive emotional state (e.g., happy, amused, admiring, relaxed, forgiving, entertained, appreciative, thankful, excited, playful, humorous, joking, optimistic, inspired, or using supportive/encouraging language).
Negative: The text contains explicit or implicit clues suggesting the speaker is (predominantly) in a negative emotional state (e.g., sad, angry, frustrated, anxious, violent, disappointed, pessimistic, critical, ironic, sarcastic, or using mocking/derogatory language).

Special Considerations:
Sarcasm/irony/mockery vs. Playful/humorous/joking:
- Negative sarcasm/mockery: Cutting, dismissive, or hostile remarks that belittle (e.g., "Oh great, another brilliant idea").
- Positive joking/playfulness: Humor intended to bond, lighten mood, or show affection (e.g., "Here comes the shining scape goat lol").

Emoticons/Slang: Use emoticons and internet slang to enhance emotional expression.
Pop Culture/Memes: Reference well-known memes, tropes, or catchphrases from Chinese web novels, games, or online culture.
Character/Plot Analysis: Include comments that analyze character motivations, plot twists, or authorial intent, as these are common in web novel discussions.

Output Format Requirements: Each comment must appear on a single line in this EXACT format with NO additional text, numbering, or bullets: SyntheticText: [comment text] Label: [positive/negative]

comment samples: 
中华传统：凡事要乘十，你再算算	Negative
搞得我又回去找了一遍鬼才。	Positive
"我用计算器算了，40一个。。。
 我的数学老师是不是也快跳出来了？"	Positive
苍老师的徒弟？	Neutral
不是4000多嘛	Neutral
卖竹鼠，三元一只，10元三只不是很正常的事情吗？	Neutral
为什么你们总觉得体育老师数学不好，我就看到过体育老师和数学老师是一个人的	Neutral
数学老师生病了，体育老师代的课	Neutral
一发40那是火车站和公园里的价格	Neutral
哈哈哈哈哈哈，盖爪	Positive
哈哈哈	Positive
每次这种竞争者桥段真的。。。。，又是必输的，还经常有，这是强行制造矛盾冲突吗？只能说水字数又弱智。一号，马杰，还有这什么螺旋，每十章出一个是吧？好伤阅读体验啊( ˘•ω•˘ )	Negative
现实这种人太多，就那袁老来说，早期研究杂交水稻的时候被同行各种打压，造谣抹黑，他老人家走后还有一群人欢呼，在外网上的评论看多了，真的会脑溢血。。	Negative
呃，我觉得你们不是同行🤣看看之前的发明	Negative
读者可悲的素质	Negative
作者真的，别这样降智，很难受的。	Negative
何止是竞争，简直是残酷。见面嘻嘻嘻，背后mmp	Negative
但是死的也快啊，什么特别行动部啊，秘密行动部之类的，一听就是死的贼快的那种部门。	Negative
千万不要带入现实世界，这本身就是个穿越到游戏世界里面的，既然是游戏那要那么多逻辑干嘛	Negative
"向往？知道为啥美女如云吗？公的都牺牲了……
 剩下的是钓鱼的。钓这些向往的……"	Negative

Paragraph for Comment Generation:
{paragraph}
"""


    try:
        time.sleep(60)
        url = "https://openrouter.ai/api/v1/chat/completions"
        
        response = requests.post(
            url=url,
            headers={"Authorization": "Bearer Insert your own api key here"},
            json={
                "model": "google/gemma-3-27b-it:free",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": 0.9,
                "max_tokens": 2000,
            },
            timeout=60,
        )
        print("Rate Limit Headers:")
        print(f"  Limit: {response.headers.get('x-ratelimit-limit')}")
        print(f"  Remaining: {response.headers.get('x-ratelimit-remaining')}")
        print(f"  Reset: {response.headers.get('x-ratelimit-reset')}")

        response.raise_for_status()
        response_data = response.json()
        response_content = response_data["choices"][0]["message"]["content"]
        # Do NOT clean line breaks; we want one comment per line
        return response_content

    except requests.exceptions.HTTPError as e:
        # Enhanced error logging
        print(f"\n=== 429 ERROR DETAILS ===")
        print(f"Status: {e.response.status_code}")
        print(f"Headers: {dict(e.response.headers)}")
        print(f"Body: {e.response.text}")
        print(f"=========================\n")
        
        with open(error_log_file_sentiment, "a", encoding="utf-8") as log_f:
            log_f.write(
                f"[{datetime.now().isoformat()}] ERROR: {repr(e)}\n"
                f"Response: {e.response.text}\n"
                f"Headers: {dict(e.response.headers)}\n\n"
            )
        return None
    
    except Exception as e:
        # Log any error to file with timestamp and short paragraph snippet
        with open(error_log_file_sentiment, "a", encoding="utf-8") as log_f:
            log_f.write(
                f"[{datetime.now().isoformat()}] ERROR during openrouter_generate_comments: {repr(e)}\n"
            )
        print(f"Error during API call: {e}")
        
        
        return None


In [15]:
# Example: generate synthetic comments for one paragraph
paragraph = """
[幽静的茶餐厅二楼。柔和的音乐声如同涓涓细流，流过人的心田。茶餐厅二楼内客人并不算多，约莫着只有十数人，三三两两，都很自觉地小声谈论着各自话题。楼梯处突然传来脚步声，令不少客人不由自主瞥一眼过去。一名穿着牛仔裤、白色polo衫的马尾辫清纯少女和一名穿着紫色休闲服的身材高挑的齐耳短发少妇，并肩走了上来。整个茶餐厅内不少人眼睛一亮！“看，两个美女！特别那个穿紫色休闲装的，啧啧，我大学在苏州呆了四年，这刚回来，没想到我们这安宜县城竟然有这么一个大美女。真是熟女啊，旁边那个虽然稚嫩些，可也清纯靓丽啊。”　“猴子，美女再好也是别人的，别再做梦了。”　“嘿，哥，别打击我嘛。对了，那个身材高挑的，齐耳短发的女的，谁啊？我活了二十多年了，这绝对是我看过所有女人中排名前三的。那五官，那气质……真的动人心魄啊。” “猴子，我告诉你，那位美女名叫‘林清’，那可是一个大人物，背景深的很，前两天我们看到的那辆价值两百万的路虎，就是她的。单单在安宜县城，她名下就有一座酒店，两座茶楼。而在我们县城这点产业，只是人家名下产业的小部分而已。” “这么厉害？”那绰号‘猴子’的青年，不由瞠目结舌。安宜县城，只是江苏境内一个普通县城，两百万的路虎在小县城内，也的确是很扎眼。　“林小姐，你的房间我们准备好了，请随我来。”茶餐厅二楼的服务员，立即迎上去。这两位美女，在服务员的带领下，在不少人的注视下步入了包厢内，而后包厢房门关闭。包厢内。　　随意点了一壶茶，便让服务员退下了。“林姐，这几天你可是天天来这，而且固定这个包厢。哈哈，你这司马昭之心，路人皆知啊。”马尾辫少女笑着瞥向窗外，在窗外对面便是一个儿童福利院。　儿童福利院的空地上，正有福利院的一些工作人员陪那些孤儿们玩耍。　]
"""

raw_output = openrouter_generate_comments(paragraph, n_comments=20)

if raw_output is None:
    print("LLM call failed; see sentiment_error_log.txt for details.")
else:
    print(raw_output[:2000])  # preview first 2000 characters

Rate Limit Headers:
  Limit: None
  Remaining: None
  Reset: None
SyntheticText: 哇！两个美女，这茶餐厅老板眼光不错啊！😍 Label: positive
SyntheticText: 紫色衣服那位姐姐也太好看了吧，气质完全是开了挂！我要恋爱了！💘 Label: positive
SyntheticText: 猴子哈哈哈哈，别做白日梦了，人家是真·大人物。😂 Label: positive
SyntheticText: 两百万的路虎！在安宜县城那是真·霸气侧漏啊！😮 Label: positive
SyntheticText: 林清？这个名字听着就很有故事感，感觉要开大后宫了！😏 Label: positive
SyntheticText: 啧啧，司马昭之心，路人皆知，这作者都暗示这么明显了。🙄 Label: negative
SyntheticText: 孤儿院...林清的背景肯定不简单啊，这剧情有点狗血。😒 Label: negative
SyntheticText: 突然觉得这茶餐厅的茶也不香了，全被这颜值冲击了。🤦‍♀️ Label: negative
SyntheticText: 作者，能不能写点别的？美女看多了也腻歪啊！😤 Label: negative
SyntheticText: 猴子估计已经开始幻想和美女们一起修炼了。笑死我了！🤣 Label: positive
SyntheticText: 安宜县城也有两百万的路虎？这作者是想显示林清的财力吧？有点过分了。🤨 Label: negative
SyntheticText: 期待林清的背景揭秘，希望是个又美又飒的女强人！😎 Label: positive
SyntheticText: 那个猴子，估计现在眼睛都离窗户不离开了，哈哈哈哈！👀 Label: positive
SyntheticText: 感觉林清对孤儿院的事情肯定有很大的参与，这剧情要反转了？🤔 Label: positive
SyntheticText: 这文笔...有点流水账的感觉，就是记录发生了什么，缺少感情描写。😐 Label: negative
SyntheticText: 林姐，快点攻略猴子，让读者们高兴一下！🙏 Label: positive
Sy

Model: google/gemma-3-27b-it:free generate synthetic data evaluation:

positive 12; all correctly labeled

negative 8; all correctly labeled

it is obvious that the generate comments each have a emoji due to the fact that in the example comment, one has an emoji. it is not effecting the overall generation quanlity of the comments, however it is something that worth noticing.